In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    Concatenate,
)
from tensorflow.keras.models import Model

In [ ]:
df = pd.read_parquet("../datasets/full_train_dataset_with_embeddings.parquet")

# feature columns
genre_cols = [
    "Action",
    "Adventure",
    "Animation",
    "Children's",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western",
    "adult"
    
]
numeric_cols = ["popularity", "vote_average", "vote_count"]
overview_embs = np.stack(df["embedding"].values).astype(np.float32)
genre_and_num = df[genre_cols + numeric_cols].to_numpy(dtype=np.float32)


In [7]:
user_vecs = df["UserID"].to_numpy(dtype=np.float32).reshape(-1, 1)
item_vecs = np.concatenate([overview_embs, genre_and_num], axis=1)
ratings = df["Rating"].to_numpy(dtype=np.float32)

In [8]:
u_train, u_test, i_train, i_test, y_train, y_test = train_test_split(
    user_vecs, item_vecs, ratings, test_size=0.2, random_state=42
)

In [ ]:
n_users = df.UserID.max() + 1  # replace with: df.UserID.max()+1
user_emb_dim = u_train.shape[1]

overview_dim = 368 
n_genres = 19  
n_numeric = 3 
movie_dim = overview_dim + n_genres + n_numeric

# ——— Inputs ———
inp_user = Input(shape=(), dtype="int32", name="user_id")
inp_movie = Input(shape=(movie_dim,), dtype="float32", name="movie_feat")

# ——— User tower ———
u = Embedding(input_dim=n_users, output_dim=user_emb_dim, name="user_emb")(inp_user)
u = Flatten()(u)
u = Dense(64, activation="relu")(u)
u = Dropout(0.2)(u)

# ——— Movie tower ———
m = Dense(256, activation="relu")(inp_movie)
m = BatchNormalization()(m)
m = Dropout(0.3)(m)
m = Dense(128, activation="relu")(m)
m = Dropout(0.2)(m)

# ——— Combine & head ———
x = Concatenate()([u, m])
x = Dense(128, activation="relu")(x)
x = Dropout(0.2)(x)
out = Dense(1, name="rating")(x)  # linear activation for regression

In [12]:
model = Model([inp_user, inp_movie], out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=[tf.keras.metrics.RootMeanSquaredError()],
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ movie_feat          │ (None, 389)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_id             │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     99,840 │ movie_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_emb            │ (None, 1)         │      6,041 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1)         │          0 │ user_emb[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        128 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     24,704 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rating (Dense)      │ (None, 1)         │        129 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 164,762 (643.60 KB)

 Trainable params: 164,250 (641.60 KB)

 Non-trainable params: 512 (2.00 KB)

In [13]:
model.fit(
    x=[u_train, i_train],
    y=y_train,
    batch_size=256,
    epochs=50,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_root_mean_squared_error", patience=5, restore_best_weights=True
        )
    ],
)

Epoch 1/50


ValueError: Input 1 of layer "functional" is incompatible with the layer: expected shape=(None, 389), found shape=(None, 405)